# Notebook 2 — Customer Propensity Scoring

**Purpose**: Build a portable 0–1 propensity score (`p_decline_like`) that places each customer on a standard↔decline axis, for use in downstream segmentation and experience analysis.

## What this notebook covers

- Feature engineering (markers × 3 states, demographics, BP flag)
- Group-aware train/test split on `application_key` to prevent same-customer leakage
- Hyperparameter sweep balancing AUC, score distribution shape, and cross-fold stability
- Final model fit and three sanity checks: score by decision class, by engine, by loading band

## Important caveat on the score
`p_decline_like` is **not** a real-world probability of being declined. The model uses `class_weight='balanced'` because declines are ~0.5% of the data — the score is a relative position on a balanced axis. A score of 0.5 means equally likely to look like either anchor population, not a 50% real-world decline probability.

## Data
Data is not included in this repository. See `data/README.md` for schema.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

RISK_MARKERS = [
    #'(Advised to) Stopping treatment',
    'Compliant with treatment',
    'History of/current high blood pressure',
    'Hospitalised previously due to blood pressure',
    'Hypertension related conditions/symptoms',
    'Knows blood pressure',
    'Need/taking blood pressure medication',
    'Requires follow-up',
    'Waiting referral or investigation'
]
MARKER_SHORT = [#'StopTx',
                'Compliant','HBP_Hx','Hosp','HBP_Related',
                'KnowsBP','Meds','Followup','WaitRef']

DEMO_COLS = [
    'age_at_application', 
    'bmi', 
    'gender', 
    'smoker', 
    'sys_pressure', 
    'dias_pressure'
]


DECISION_ORDER = ['standard','non-standard','refer/evidence_required','postpone','decline']
DECISION_PAL   = {
    'standard':                "#86d6a9",
    'non-standard':            "#FFF0A0",
    'refer/evidence_required': "#FF9AB3",
    'postpone':                "#57a4d7",
    'decline':                 "#b383d1",
}


In [ ]:
# Load the dataset
df = pd.read_csv("../data/FULL_LIFE_TABLE.csv")

# Print summary statistics
print(f"Rows         : {len(df):,}")
print(f"Applications : {df['application_key'].nunique():,}")
print(f"Customers    : {df['customer_key'].nunique():,}")
print(f"Engines      : {df['enquiry_engine'].nunique()}")
print(f"\nDecision distribution:")
print(df['decision'].value_counts().to_string())

In [ ]:
# Inspect missing values in demographic columns
for col in DEMO_COLS:
    print(f"{col:<25}: {df[col].isna().sum():,} NaN")

# Rows where ALL demographics are NaN
all_missing = df[DEMO_COLS].isna().all(axis=1)
print(f"\nRows with all demographics NaN: {all_missing.sum():,}\n")

# Inspect missing values in risk markers
for col in RISK_MARKERS:
    print(f"{col:<25}: {df[col].isna().sum():,} NaN")

# Rows where ALL risk markers are NaN
all_missing = df[RISK_MARKERS].isna().all(axis=1)
print(f"\nRows with all risk markers NaN: {all_missing.sum():,}")

In [ ]:
df.columns

# Feature engineering

In [ ]:
def build_features(src):
    """Build the full feature matrix: markers + demographics + engine + date.

    BP handling: dias_pressure / sys_pressure are overloaded — 'F' means BP not
    known, otherwise it's a numeric string. We split them into a numeric BP
    feature (median-imputed when missing) plus an explicit bp_known flag, so
    the model can learn whether 'BP is unknown' is itself a signal.
    """
    # Markers — one-hot, all 3 states (T/F/NAsk)
    rm = pd.get_dummies(src[RISK_MARKERS], drop_first=False)

    # Convert overloaded BP columns to numeric (NaN where 'F')
    sys_raw  = pd.to_numeric(src['sys_pressure'],  errors='coerce')
    dias_raw = pd.to_numeric(src['dias_pressure'], errors='coerce')

    # Median over rows that have a BP reading
    sys_median  = sys_raw.median()
    dias_median = dias_raw.median()

    # Continuous demographics + date as ordinal
    cont = pd.DataFrame({
        'age':      pd.to_numeric(src['age_at_application'], errors='coerce'),
        'bmi':      pd.to_numeric(src['bmi'],                errors='coerce'),
        'sys':      sys_raw.fillna(sys_median),
        'dias':     dias_raw.fillna(dias_median),
        'date_ord': pd.to_numeric(src['year_month_cont'],    errors='coerce'),
    }, index=src.index)

    # Binary flags
    bin_demo = pd.DataFrame({
        'gender_M': (src['gender'] == 'Male').astype(int),
        'smoker_T': (src['smoker'] == 'Yes').astype(int),
        'bp_known': sys_raw.notna().astype(int),  # 1 = BP value provided, 0 = 'F'
    }, index=src.index)

    # Engine — one-hot
    eng = pd.get_dummies(src['enquiry_engine'], prefix='engine', drop_first=False)

    return pd.concat([rm, cont, bin_demo, eng], axis=1).fillna(0)


X = build_features(df)
print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} columns\n")

marker_cols = [c for c in X.columns if any(c.startswith(m + '_') for m in RISK_MARKERS)]
engine_cols = [c for c in X.columns if c.startswith('engine_')]
cont_cols   = ['age','bmi','sys','dias','date_ord']
bin_cols    = ['gender_M','smoker_T']

print("Feature breakdown:")
print(f"  Risk marker dummies (T/F/NAsk × 9 markers) : {len(marker_cols):>3}")
print(f"  Continuous demographics + date ordinal     : {len(cont_cols):>3}")
print(f"  Binary demographics + bp_known             : {len(bin_cols)+1:>3}") 
print(f"  Engine dummies                             : {len(engine_cols):>3}")
print(f"  TOTAL                                      : {X.shape[1]:>3}")

# Prepare the test/train sets

In [ ]:
# Anchor masks -> we only train on standard & decline decisions
mask_std = (df['decision'] == 'standard').values
mask_dec = (df['decision'] == 'decline').values
mask_anchor = mask_std | mask_dec

X_anchor = X[mask_anchor].reset_index(drop=True)
y_anchor = mask_dec[mask_anchor].astype(int)
groups_anchor = df.loc[mask_anchor, 'application_key'].values

print(f"Anchor set:")
print(f"  Standards : {mask_std.sum():,}")
print(f"  Declines  : {mask_dec.sum():,}  ({mask_dec.sum() / mask_anchor.sum() * 100:.2f}% prevalence)")

# Group-aware split (no application appears in both train and test)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(X_anchor, y_anchor, groups=groups_anchor))

X_tr, X_te = X_anchor.iloc[tr_idx], X_anchor.iloc[te_idx]
y_tr, y_te = y_anchor[tr_idx], y_anchor[te_idx]

print(f"\nGroup-aware split:")
print(f"  Train: {len(tr_idx):,} rows from {pd.Series(groups_anchor[tr_idx]).nunique():,} apps")
print(f"  Test : {len(te_idx):,} rows from {pd.Series(groups_anchor[te_idx]).nunique():,} apps")
print(f"  Train/test app overlap: {len(set(groups_anchor[tr_idx]) & set(groups_anchor[te_idx]))}")

# Build the Random Forest

Baseline parameters

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=50, max_depth=3, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1,
    max_features = 1)

rf_base.fit(X_tr, y_tr)
auc_rf_base = roc_auc_score(y_te, rf_base.predict_proba(X_te)[:,1])

print("Group-aware AUCs (standards vs declines):")
print(f"  Random forest           : {auc_rf_base:.3f}")
print()
    

In [ ]:
# Random forest OVERFITTING
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
auc_rf = roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])

print("Group-aware AUCs (standards vs declines):")
print(f"  Random forest           : {auc_rf:.3f}")
print()

# Hyperparameter tuning for `p_decline_like`

The model above (deep RF, all features) hits AUC ~0.99. That's not leakage —
it's what deep RFs do on 1% prevalence balanced classification. But high AUC
doesn't mean a useful **segmentation score**: a deep RF produces a bimodal
score distribution (piles at ~0.05 and ~0.95, empty middle), which makes
"top X% by risk" cutoffs land in gaps.

So tuning isn't optimising AUC — it's picking a config that gives a usable
0–1 score for cross-engine experience analysis.

**Three-part objective**:
1. **AUC** — must be reasonable (≥0.92), but not the goal
2. **Distribution shape** — score should spread across [0,1], not collapse
   to extremes. Measured by *decile coverage* (how many of 10 deciles hold
   ≥1% of population) and *score Gini* (lower = more spread)
3. **Stability across folds** — same customer should get a similar score
   from models trained on different group-aware folds

These objectives **fight each other** — the deepest RF wins on AUC but loses
on distribution. We're picking a frontier point, not a maximum.

Feature set: full `X` (markers + demographics + date + engine). Engine and
date stay in because they're legitimate signal — the question of "should the
score be portable across engines" is a separate decision we can make at the
*application* stage by either dropping those features or accepting the
score is engine-conditional.


## Distribution metrics helper

In [ ]:
def score_distribution_metrics(scores, n_bins=10):
    """Decile coverage and Gini of the score histogram.

    decile_coverage : number of deciles containing >= 1% of population (max 10)
    score_gini      : Gini coefficient of histogram counts.
                      0 = perfectly uniform spread, 1 = all mass in one bin.
    """
    edges = np.linspace(0, 1, n_bins + 1)
    counts, _ = np.histogram(scores, bins=edges)
    pct = counts / counts.sum()
    decile_coverage = int((pct >= 0.01).sum())
    sorted_pct = np.sort(pct)
    n = len(sorted_pct)
    cum = np.cumsum(sorted_pct)
    gini = (2 * np.sum(np.arange(1, n+1) * sorted_pct) - (n + 1) * cum[-1]) / (n * cum[-1])
    return decile_coverage, gini, pct


def print_distribution(scores, label):
    """Print AUC-independent distribution diagnostics."""
    cov, gini, pct = score_distribution_metrics(scores)
    print(f"{label}")
    print(f"  Decile coverage         : {cov}/10")
    print(f"  Score-distribution Gini : {gini:.3f}  (lower = more spread)")
    print(f"  Per-decile share (%):")
    for i, p in enumerate(pct):
        bar = '#' * int(p * 200)
        print(f"    [{i/10:.1f}-{(i+1)/10:.1f}] : {p*100:5.2f}%  {bar}")


In [ ]:
# Sanity check on the deep RF you already trained
p_full_deep = rf.predict_proba(X)[:, 1]
print_distribution(p_full_deep, f"Deep RF (depth=None, leaf=20)  -  AUC={auc_rf:.3f}")


## Hyperparameter sweep

Focused grid on the parameters that drive the AUC ↔ distribution trade-off:

- `max_depth` ∈ {4, 6, 8, 12, None}
- `min_samples_leaf` ∈ {20, 100, 500, 2000}
- `class_weight` ∈ {'balanced', None}
- `n_estimators`: fixed at 200

For each config: AUC on the held-out test fold + decile coverage + Gini on
the full-population score. Stability test (slow) comes after we shortlist.


### Reasons for grid search
1. The search space is small and the parameters are interpretable. When you only have 3 dimensions and you already know which values matter (shallow vs deep, small vs large leaves, balanced vs not), a grid lets you read the shape of the trade-off — "depth 8 with leaf 100 vs depth 8 with leaf 500" is a meaningful comparison you can see directly. Random search and Bayesian methods give you a single best config but obscure the topology.
2. The objective isn't a single scalar. Bayesian optimisation needs a single number to maximise. Our objective has three components that fight each other (AUC ↑, decile coverage ↑, Gini ↓), and the right answer is somewhere on the frontier. You can collapse them into a weighted sum but the weights are arbitrary — easier to show all three columns in a table and pick by inspection.
3. 40 fits is cheap enough. Maybe 15–20 min total. Not worth the engineering of setting up Optuna or sklearn's RandomizedSearchCV for this.

**Few things to note**
- No cross-validation in the sweep itself. Each config is evaluated on the single 80/20 group-aware split you already made. CV happens later, only on the shortlisted 3–4 configs in the stability test (GroupKFold(n_splits=5)). This is a deliberate trade-off — running 5-fold CV on 40 configs would be 200 fits.
- Frozen n_estimators=200. More trees mostly affects stability, not AUC, after ~100. The stability test fixes this at 200 too. If you wanted to sweep it, you'd add it as a 4th grid dimension.
- No max_features sweep. Default is sqrt(n_features) which is fine for this data.
- The shortlist → stability test is the manual second stage. That's where the real validation happens — picking 3–4 candidates from the table and CV-validating them.



In [ ]:
from itertools import product
import time
from sklearn.metrics import average_precision_score


grid = list(product(
    [4, 6, 8, 12, None],          # max_depth
    [20, 100, 500, 2000],         # min_samples_leaf
    ['balanced', None],           # class_weight
))

results = []
print(f"Sweeping {len(grid)} configs...\n")
t0 = time.time()

for i, (md_, msl, cw) in enumerate(grid, 1):
    rf_i = RandomForestClassifier(
        n_estimators=200, max_depth=md_, min_samples_leaf=msl,
        class_weight=cw, random_state=42, n_jobs=-1)
    rf_i.fit(X_tr, y_tr)
    auc_i = roc_auc_score(y_te, rf_i.predict_proba(X_te)[:, 1])
    ap_i = average_precision_score(y_te, rf_i.predict_proba(X_te)[:, 1])
    p_full_i = rf_i.predict_proba(X)[:, 1]
    cov_i, gini_i, _ = score_distribution_metrics(p_full_i)
    results.append({
        'max_depth': md_ if md_ is not None else 'None',
        'min_samples_leaf': msl,
        'class_weight': cw if cw is not None else 'None',
        'auc': round(auc_i, 4),
        'average_precision': round(ap_i, 4),
        'decile_coverage': cov_i,
        'score_gini': round(gini_i, 3),
    })
    elapsed = time.time() - t0
    print(f"[{i:>2}/{len(grid)}] depth={str(md_):>4}  leaf={msl:>4}  cw={str(cw):>8} "
          f"->  AUC={auc_i:.3f}  cov={cov_i}/10  gini={gini_i:.2f}  ({elapsed:.0f}s)")

results_df = pd.DataFrame(results).sort_values(
    ['decile_coverage', 'auc'], ascending=[False, False])
print("\nTop 15 by (coverage desc, AUC desc):")
print(results_df.head(15).to_string(index=False))


**AUC** : Higher AUC means the model is better at separating the anchor groups

**Decile coverage**: Asks how many of the 10 score buckets actually have customers in them.
- since every customer gets a number between 0 and 1,  their `p_decline_like`, we want to check the shape of that distribution. 
- So we slice the scale into 10 buckets and count how many customers fall into each
- we want all buckets to have at least 1% of the customers

In [ ]:
# Visualise the trade-off
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for cw, sub in results_df.groupby('class_weight'):
    ax[0].scatter(sub['auc'], sub['decile_coverage'], label=f'cw={cw}', s=80, alpha=0.7)
    ax[1].scatter(sub['auc'], sub['score_gini'], label=f'cw={cw}', s=80, alpha=0.7)

ax[0].set_xlabel('AUC (test fold)'); ax[0].set_ylabel('Decile coverage (/10)')
ax[0].set_title('AUC vs distribution spread\n(higher coverage = score uses more of [0,1])')
ax[0].grid(alpha=0.3); ax[0].legend()

ax[1].set_xlabel('AUC (test fold)'); ax[1].set_ylabel('Score-distribution Gini')
ax[1].set_title('AUC vs concentration\n(lower Gini = more spread out)')
ax[1].grid(alpha=0.3); ax[1].legend()

plt.tight_layout(); plt.show()


## Stability test on shortlisted configs

Pick 3–4 candidates from the sweep that look like a good AUC/distribution
trade-off, then test stability: refit on 5 group-aware folds, score every
customer with each fold's model, report mean per-customer std of the score.

**Lower mean std = more stable score = more reliable segments.**

Edit `shortlist` below to your top candidates from the table above.


In [ ]:
# EDIT to your shortlist after looking at the sweep results
shortlist = [
    {'max_depth': 6,    'min_samples_leaf': 500,  'class_weight': 'balanced'},
    {'max_depth': 8,    'min_samples_leaf': 100,  'class_weight': 'balanced'},
    {'max_depth': 12,   'min_samples_leaf': 100,  'class_weight': 'balanced'},
    {'max_depth': None, 'min_samples_leaf': 20,   'class_weight': 'balanced'},  # baseline
]


In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

stability_results = []
for cfg in shortlist:
    cfg_label = f"depth={cfg['max_depth']}, leaf={cfg['min_samples_leaf']}, cw={cfg['class_weight']}"
    print(f"\n>>> {cfg_label}")
    fold_scores = np.zeros((len(X), 5))
    fold_aucs = []
    for k, (tr_k, te_k) in enumerate(gkf.split(X_anchor, y_anchor, groups=groups_anchor)):
        rf_k = RandomForestClassifier(
            n_estimators=200,
            max_depth=cfg['max_depth'],
            min_samples_leaf=cfg['min_samples_leaf'],
            class_weight=cfg['class_weight'],
            random_state=42 + k, n_jobs=-1)
        rf_k.fit(X_anchor.iloc[tr_k], y_anchor[tr_k])
        auc_k = roc_auc_score(y_anchor[te_k], rf_k.predict_proba(X_anchor.iloc[te_k])[:, 1])
        fold_aucs.append(auc_k)
        fold_scores[:, k] = rf_k.predict_proba(X)[:, 1]
        print(f"  fold {k+1}: AUC={auc_k:.3f}")

    per_customer_std = fold_scores.std(axis=1)
    mean_std = per_customer_std.mean()
    rep_scores = fold_scores.mean(axis=1)
    cov, gini, _ = score_distribution_metrics(rep_scores)

    stability_results.append({
        'config': cfg_label,
        'mean_auc': round(np.mean(fold_aucs), 3),
        'auc_std': round(np.std(fold_aucs), 4),
        'mean_score_std': round(mean_std, 4),
        'decile_coverage': cov,
        'score_gini': round(gini, 3),
    })
    print(f"  Mean AUC across folds        : {np.mean(fold_aucs):.3f} +/- {np.std(fold_aucs):.4f}")
    print(f"  Mean per-customer score std  : {mean_std:.4f}  (lower = more stable)")
    print(f"  Decile coverage              : {cov}/10")
    print(f"  Score Gini                   : {gini:.3f}")

stability_df = pd.DataFrame(stability_results)
print("\n\n=== Stability summary ===")
print(stability_df.to_string(index=False))


## Final fit + sanity checks

Pick the winning config, refit on the full anchor population, score the full
dataset, and run three diagnostic plots:

1. Score distribution by decision class (expect: standards low, declines high)
2. Score by engine (expect: similar shapes if score is portable)
3. Score by em_band within non-standards (expect: roughly flat — confirms
   finding 6.1 from a different angle)


In [ ]:
# EDIT after seeing stability results
FINAL_CFG = {'max_depth': 8, 'min_samples_leaf': 100, 'class_weight': 'balanced'}

rf_final = RandomForestClassifier(
    n_estimators=200,
    max_depth=FINAL_CFG['max_depth'],
    min_samples_leaf=FINAL_CFG['min_samples_leaf'],
    class_weight=FINAL_CFG['class_weight'],
    random_state=42, n_jobs=-1)
rf_final.fit(X_anchor, y_anchor)

df['p_decline_like'] = rf_final.predict_proba(X)[:, 1]
print(f"Scored {len(df):,} rows.")
print(df['p_decline_like'].describe())


In [ ]:
# Sanity check 1: score distribution by decision (preferred plot style)
df_scored = df.copy().reset_index(drop=True)

print("Mean predicted score by decision:\n")
score_summary = (df_scored.groupby('decision', observed=True)['p_decline_like']
                          .agg(['mean','median','count']).round(3))
print(score_summary.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 41)
for d in DECISION_ORDER:
    if d not in df_scored['decision'].unique():
        continue
    sub = df_scored.loc[df_scored['decision'] == d, 'p_decline_like']
    ax.hist(sub, bins=bins, alpha=0.45, color=DECISION_PAL[d],
            label=f'{d} (n={len(sub):,})', density=True)
ax.set_xlabel('p_decline_like')
ax.set_ylabel('Density')
ax.set_title('Score distribution by decision - all decision types', fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Sanity check 2: score by engine (top 10 by volume)
top_engines = df['enquiry_engine'].value_counts().head(10).index.tolist()
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=df[df['enquiry_engine'].isin(top_engines)],
    x='enquiry_engine', y='p_decline_like',
    order=top_engines, ax=ax, showfliers=False)
ax.set_title("p_decline_like by engine (top 10 by volume)\n"
             "similar shapes = portable score; very different shapes = engine-conditional")
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


In [ ]:
# Sanity check 3: within non-standards, score by em_band
ns = df[df['decision'] == 'non-standard'].copy()
ns['em_load_num'] = pd.to_numeric(ns['em_load'], errors='coerce')
ns = ns.dropna(subset=['em_load_num'])
ns['em_band'] = pd.cut(
    ns['em_load_num'],
    bins=[-np.inf, 35, 60, 101, np.inf],
    labels=['Light (<=35)', 'Moderate (<=60)', 'Heavy (<=101)', 'Very Heavy (>101)'])

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=ns, x='em_band', y='p_decline_like',
    order=['Light (<=35)', 'Moderate (<=60)', 'Heavy (<=101)', 'Very Heavy (>101)'],
    ax=ax, showfliers=False)
ax.set_title("p_decline_like within non-standards, by em_band\n"
             "expect: roughly flat - em_load and marker-based risk decoupled (finding 6.1)")
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print("\nMedian p_decline_like by em_band:")
print(ns.groupby('em_band')['p_decline_like'].agg(['median','mean','count']).round(3))


In [ ]:
df.head()

In [ ]:
df.drop(columns="Unnamed: 0").to_csv("../data/FULL_WITH_SCORES.csv", index = False  )

In [ ]:
df.columns

In [ ]:
df.info()